In [9]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, ToolMessage

# 1. Define three Tools

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def subtract_numbers(a: int, b: int) -> int:
    """Subtract the second number from the first."""
    return a - b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


# 2. Initialize LLM
llm = ChatOllama(model="qwen2.5:1.5b")


# 3. Bind all three tools
tools = [add_numbers, subtract_numbers, multiply_numbers]

llm_with_tools = llm.bind_tools(tools)


# 4. Ask a question, 
# create a message array which will record all conversation history, 
# including tool calls and their results
messages = [
    HumanMessage(content="Multiply 10 and 20 using the appropriate tool. then substract 5 an 2, then play ping pong")
]


# 5. LLM decides which tool to call
ai_message = llm_with_tools.invoke(messages)

print("AI Tool Calls:", ai_message.tool_calls)


# 6. Execute the requested tool(s)
messages.append(ai_message)

print("messages:", messages)

# Iterate through every tool-call request made by the LLM
for tool_call in ai_message.tool_calls:

    # Find the actual Python function corresponding to the tool
    # selected by the LLM
    selected_tool = None

    for tool in tools:
        if tool.name == tool_call["name"]:
            selected_tool = tool
            break

    # Execute the selected tool by passing the arguments
    # generated by the LLM and capture its result
    result = selected_tool.invoke(tool_call["args"])

    # Create a ToolMessage containing the execution result
    # Link it to the original tool request using its unique ID
    tool_message = ToolMessage(
        content=str(result),
        tool_call_id=tool_call["id"]
    )

    # Add the tool's result to the conversation history
    # so the LLM can use it to generate its final response
    messages.append(tool_message)



# 8. Generate final response
final_response = llm.invoke(messages)

print(final_response.content)

AI Tool Calls: [{'name': 'multiply_numbers', 'args': {'a': 10, 'b': 20}, 'id': '8b8af26e-adfd-4aef-87f0-9f726ed01734', 'type': 'tool_call'}, {'name': 'subtract_numbers', 'args': {'a': 25, 'b': 5}, 'id': 'e9d227fc-0837-4fbb-91a2-b46483eda4ca', 'type': 'tool_call'}]
messages: [HumanMessage(content='Multiply 10 and 20 using the appropriate tool. then substract 5 an 2, then play ping pong', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-11-03T10:41:40.8473301Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3314660900, 'load_duration': 7117600, 'prompt_eval_count': 265, 'prompt_eval_duration': 258238000, 'eval_count': 55, 'eval_duration': 2798565000, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a1a100-0cfb-74e0-8ed4-0784d28eae3d-0', tool_calls=[{'name': 'multiply_numbers', 'args': {'a': 10, 'b': 20}, 'id': '8b8af26e-adfd-4aef-87f0-